# Notebook 02 — Chuẩn bị dữ liệu cho Chương 4

Tạo bộ dữ liệu cố định (seed cố định, chạy lại ra đúng bộ cũ) gồm ba phần:

| Phần | Số lượng | Nguồn | Dùng cho |
|---|---|---|---|
| Ảnh đời thường | 100 ảnh, ~5 câu tham chiếu/ảnh | COCO **val2017** | So sánh 3 mô hình; CIDEr, CLIPScore |
| Figure khoa học | 40 hình, kèm caption gốc + đoạn văn nhắc đến hình + chữ OCR | SciCap+ (tập test) | Cải tiến bằng ngữ cảnh (2A) |
| Biểu đồ tự vẽ | 20 biểu đồ, nửa có ghi số | Sinh bằng code, có đáp án | Đếm lỗi sai dữ kiện; tự kiểm tra (2B) |

**Không cần GPU.** Runtime mặc định (CPU) là đủ.

Dữ liệu được lưu vào **Google Drive** để notebook 03 dùng lại sau khi phiên Colab kết thúc. Giấy phép: COCO theo giấy phép từng ảnh (đã lưu kèm), SciCap+ CC BY-NC 4.0 — chỉ dùng cho học tập, **không đưa ảnh lên GitHub**.

## 1. Cấu hình

In [ ]:
SEED = 2026
N_COCO = 100
N_SCICAP = 40
N_CHART_PER_KIND = 5          # 5 × (cột, đường) × (có số, không số) = 20 biểu đồ
USE_DRIVE = True              # lưu vào Google Drive để notebook sau dùng lại
DOWNLOAD_FULL_SCICAP_IMG = False  # chỉ bật nếu cách đọc từ xa ở mục 4 thất bại (tải 18 GB)
# Chỉ lấy các loại hình gần với biểu đồ; mục 4 sẽ in danh sách loại có trong dữ liệu.
SCICAP_TYPES = {"Graph Plot", "Bar Chart", "Scatterplot"}

## 2. Thư mục lưu và thư viện

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/ImageCaptioning/data"
else:
    DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)
print("Lưu dữ liệu vào:", DATA_DIR)

!pip install -q remotezip

## 3. COCO val2017: chọn ảnh và câu tham chiếu

In [ ]:
import json, random, time, urllib.request
from collections import defaultdict
from remotezip import RemoteZip

# Chỉ đọc file chú thích cần dùng trong gói 241 MB, không tải cả gói.
with RemoteZip("http://images.cocodataset.org/annotations/annotations_trainval2017.zip") as z:
    coco = json.loads(z.read("annotations/captions_val2017.json"))

refs = defaultdict(list)
for ann in coco["annotations"]:
    refs[ann["image_id"]].append(ann["caption"].strip())
images = {img["id"]: img for img in coco["images"]}
licenses = {lic["id"]: lic["name"] for lic in coco["licenses"]}

coco_ids = random.Random(SEED).sample(sorted(images), N_COCO)
coco_dir = os.path.join(DATA_DIR, "coco")
os.makedirs(coco_dir, exist_ok=True)


def download(url, path, tries=4):
    for attempt in range(tries):
        try:
            if not os.path.exists(path):
                urllib.request.urlretrieve(url, path)
            return
        except Exception as error:
            print(f"  thử lại {attempt + 1}: {error}")
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"Không tải được {url}")


coco_items = []
for image_id in coco_ids:
    img = images[image_id]
    download(img["coco_url"], os.path.join(coco_dir, img["file_name"]))
    coco_items.append({
        "id": f"coco_{image_id}", "source": "coco_val2017", "file": f"coco/{img['file_name']}",
        "width": img["width"], "height": img["height"], "references": refs[image_id],
        "license": licenses[img["license"]], "flickr_url": img.get("flickr_url"),
    })
print(len(coco_items), "ảnh COCO; số câu tham chiếu:", sorted({len(i["references"]) for i in coco_items}))
print(coco_items[0]["file"], coco_items[0]["references"][:2])

## 4. SciCap+: figure khoa học kèm caption gốc, đoạn văn nhắc đến hình và chữ OCR

Tải trọn hai gói nhỏ (caption ~394 MB, đoạn văn ~433 MB) vào ổ tạm của Colab, rồi chỉ đọc các ảnh cần dùng từ gói ảnh 18 GB qua mạng.

In [ ]:
import subprocess

SCICAP = "https://huggingface.co/datasets/anselyang/SciCapPlus/resolve/main"
TMP = "/content/scicap"
os.makedirs(TMP, exist_ok=True)


def fetch(name):
    # -c: tải tiếp nếu bị ngắt giữa chừng
    subprocess.run(["wget", "-q", "-c", "-O", f"{TMP}/{name}", f"{SCICAP}/{name}"], check=True)
    print(name, round(os.path.getsize(f"{TMP}/{name}") / 1e6), "MB")


for name in ["captions.zip", "mentions_paragraph.zip"]:
    fetch(name)

In [ ]:
import zipfile
from collections import Counter

cap_zip = zipfile.ZipFile(f"{TMP}/captions.zip")
test_files = [n for n in cap_zip.namelist() if n.startswith("captions/test/") and n.endswith(".json")]
records = [json.loads(cap_zip.read(n)) for n in test_files]
print(len(records), "hình trong tập test")
print("Các loại hình:", Counter(r["figure-type"] for r in records).most_common())

In [ ]:
import io, pickle, re
import numpy as np

# Đoạn văn được lưu dạng .npy. Không dùng allow_pickle=True vì có thể chạy mã tùy ý;
# nếu file là pickle, chỉ cho phép dựng lại mảng numpy và chuỗi ký tự.
SAFE = {("numpy.core.multiarray", "_reconstruct"), ("numpy._core.multiarray", "_reconstruct"),
        ("numpy", "ndarray"), ("numpy", "dtype"),
        ("numpy.core.multiarray", "scalar"), ("numpy._core.multiarray", "scalar")}


class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if (module, name) in SAFE:
            return super().find_class(module, name)
        raise pickle.UnpicklingError(f"Không cho phép {module}.{name}")


def flatten(value):
    if isinstance(value, dict):
        # File của SciCap+ có dạng {"caption": ..., "mentions": [...]}: CHỈ lấy đoạn văn,
        # tuyệt đối không lấy "caption" vì đó là đáp án (caption gốc của hình).
        return flatten(value.get("mentions", []))
    if isinstance(value, (str, bytes)):
        return [value.decode("utf8", "ignore") if isinstance(value, bytes) else value]
    if isinstance(value, np.ndarray):
        return flatten(value.tolist())  # mảng 0 chiều trả về đúng đối tượng bên trong (có thể là dict)
    if isinstance(value, (list, tuple)):
        return [s for v in value for s in flatten(v)]
    return [str(value)]


def load_mentions(raw):
    try:
        return flatten(np.load(io.BytesIO(raw), allow_pickle=False))
    except ValueError:
        buffer = io.BytesIO(raw)
        version = np.lib.format.read_magic(buffer)
        if version == (1, 0):
            np.lib.format.read_array_header_1_0(buffer)
        else:
            np.lib.format.read_array_header_2_0(buffer)
        return flatten(SafeUnpickler(buffer).load())


men_zip = zipfile.ZipFile(f"{TMP}/mentions_paragraph.zip")
men_names = set(men_zip.namelist())
sample = next(n for n in men_names if n.startswith("mentions_paragraph/test/") and n.endswith(".npy"))
print("Ví dụ file đoạn văn:", sample)
print(load_mentions(men_zip.read(sample))[:2])

FIG_PREFIX = re.compile(r"^\s*(?:Figure|Fig\.?)\s*\d+[a-zA-Z]?\s*[:.]?\s+", re.IGNORECASE)


def clean_caption(text):
    """Bỏ tiền tố "Figure 3:" khỏi caption gốc, như bước chuẩn hóa của SciCap."""
    return FIG_PREFIX.sub("", text).strip()

In [ ]:
def mention_name(figure_id):
    return f"mentions_paragraph/test/{figure_id.rsplit('.', 1)[0]}_mentions.npy"


candidates = []
for r in records:
    caption = clean_caption(r["0-originally-extracted"])
    n_words = len(caption.split())
    if r["contains-subfigure"] or r["figure-type"] not in SCICAP_TYPES or not 5 <= n_words <= 60:
        continue
    if mention_name(r["figure-ID"]) not in men_names:
        continue
    candidates.append(r)
print(len(candidates), "hình đạt điều kiện")

rng = random.Random(SEED)
rng.shuffle(candidates)
picked = []
for r in candidates:
    paragraphs = [p.strip() for p in load_mentions(men_zip.read(mention_name(r["figure-ID"]))) if p.strip()]
    caption = clean_caption(r["0-originally-extracted"])
    # Loại hình nếu đoạn văn vẫn chứa nguyên caption (tránh lộ đáp án cho cấu hình B)
    if paragraphs and caption.lower()[:50] not in " ".join(paragraphs).lower():
        picked.append((r, paragraphs))
    if len(picked) == N_SCICAP:
        break
print(len(picked), "hình được chọn; loại:", Counter(r["figure-type"] for r, _ in picked))

In [ ]:
from remotezip import RemoteZip

scicap_dir = os.path.join(DATA_DIR, "scicap")
os.makedirs(scicap_dir, exist_ok=True)


def open_remote(url, tries=5):
    for attempt in range(tries):
        try:
            return RemoteZip(url)
        except Exception as error:
            print(f"  mở lại {attempt + 1}: {error}")
            time.sleep(5 * (attempt + 1))
    raise RuntimeError("Không đọc được mục lục của img.zip; hãy bật DOWNLOAD_FULL_SCICAP_IMG ở mục 1")


if DOWNLOAD_FULL_SCICAP_IMG:
    fetch("img.zip")
    img_zip = zipfile.ZipFile(f"{TMP}/img.zip")
else:
    img_zip = open_remote(f"{SCICAP}/img.zip")
by_name = {n.rsplit("/", 1)[-1]: n for n in img_zip.namelist() if not n.endswith("/")}
print(len(by_name), "ảnh trong img.zip; ví dụ:", next(iter(by_name.values())))

scicap_items = []
for r, paragraphs in picked:
    member = by_name.get(r["figure-ID"])
    if member is None:
        print("  không tìm thấy ảnh:", r["figure-ID"])
        continue
    path = os.path.join(scicap_dir, r["figure-ID"])
    if not os.path.exists(path):
        with open(path, "wb") as f:
            f.write(img_zip.read(member))
    scicap_items.append({
        "id": f"scicap_{r['figure-ID'].rsplit('.', 1)[0]}", "source": "scicap_plus_test",
        "file": f"scicap/{r['figure-ID']}", "figure_type": r["figure-type"],
        "references": [clean_caption(r["0-originally-extracted"])],
        "mention_paragraphs": paragraphs, "ocr_text": " ".join(r.get("Img-text") or []),
        "paper_id": r["paper-ID"],
    })
print(len(scicap_items), "hình SciCap+ đã lưu")
leaks = [it["id"] for it in scicap_items if it["references"][0].lower()[:50] in " ".join(it["mention_paragraphs"]).lower()]
print("Kiểm tra lộ đáp án (phải bằng 0):", len(leaks))
print(json.dumps(scicap_items[0], ensure_ascii=False)[:800])

## 5. Biểu đồ tự vẽ có đáp án

In [ ]:
import json
import os
import random

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Mỗi chủ đề đi kèm nhãn trục và miền giá trị phù hợp, để biểu đồ trông hợp lý.
BAR_SPECS = [
    ("Caption accuracy of three models", "Model", "Accuracy (%)", ["Model A", "Model B", "Model C"], (40, 95)),
    ("CIDEr score of captioning models", "Model", "CIDEr", ["BLIP", "BLIP-2", "LLaVA"], (60, 150)),
    ("GPU memory usage", "Model", "Memory (GB)", ["Model A", "Model B", "Model C", "Model D"], (2, 16)),
    ("Images per dataset split", "Split", "Images (thousands)", ["Train", "Val", "Test"], (5, 120)),
]
LINE_SPECS = [
    ("Number of papers per year", "Year", "Papers", "year", "up"),
    ("Validation accuracy by epoch", "Epoch", "Accuracy (%)", "epoch", "up"),
    ("Training loss by epoch", "Epoch", "Loss", "epoch", "down"),
    ("Monthly active users", "Month", "Users (thousands)", "month", "mixed"),
]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May"]


def trend_of(values):
    diffs = [b - a for a, b in zip(values, values[1:])]
    if all(d > 0 for d in diffs):
        return "increasing"
    if all(d < 0 for d in diffs):
        return "decreasing"
    return "mixed"


def make_charts(out_dir, n_per_kind=5, seed=2026):
    """Vẽ biểu đồ cột và đường, một nửa ghi số trên dữ liệu, kèm đáp án đúng."""
    rng = random.Random(seed)
    os.makedirs(out_dir, exist_ok=True)
    items = []
    for kind in ("bar", "line"):
        for labelled in (True, False):
            for k in range(n_per_kind):
                if kind == "bar":
                    title, xlabel, ylabel, xs, (lo, hi) = rng.choice(BAR_SPECS)
                    values = rng.sample(range(lo, hi + 1), len(xs))
                else:
                    title, xlabel, ylabel, axis, shape = rng.choice(LINE_SPECS)
                    if axis == "year":
                        start = rng.choice([2019, 2020, 2021])
                        xs = [str(start + i) for i in range(5)]
                    elif axis == "epoch":
                        xs = [str(i) for i in range(1, 6)]
                    else:
                        xs = MONTHS
                    v = rng.randint(30, 60) if shape == "down" else rng.randint(10, 40)
                    values = [v]
                    for _ in range(4):
                        step = {"up": rng.randint(3, 15), "down": -rng.randint(1, 6), "mixed": rng.randint(-10, 10)}[shape]
                        v = max(1, v + step)
                        values.append(v)
                fig, ax = plt.subplots(figsize=(6, 4))
                if kind == "bar":
                    bars = ax.bar(xs, values, color="#4C78A8")
                    if labelled:
                        ax.bar_label(bars, labels=[str(v) for v in values], padding=2)
                else:
                    ax.plot(xs, values, marker="o", color="#4C78A8")
                    if labelled:
                        for x, v in zip(xs, values):
                            ax.annotate(str(v), (x, v), textcoords="offset points", xytext=(0, 6), ha="center")
                ax.set_title(title)
                ax.set_xlabel(xlabel)
                ax.set_ylabel(ylabel)
                ax.set_ylim(0, max(values) * 1.2)
                name = f"chart_{kind}_{'labelled' if labelled else 'unlabelled'}_{k:02d}.png"
                fig.savefig(os.path.join(out_dir, name), dpi=150, bbox_inches="tight")
                plt.close(fig)
                items.append({
                    "id": name[:-4], "source": "synthetic_chart", "file": name,
                    "chart_type": kind, "values_labelled": labelled,
                    "title": title, "x_label": xlabel, "y_label": ylabel,
                    "categories": xs, "values": values,
                    "max": xs[values.index(max(values))], "min": xs[values.index(min(values))],
                    "trend": trend_of(values) if kind == "line" else None,
                })
    return items


chart_items = make_charts(os.path.join(DATA_DIR, "charts"), n_per_kind=N_CHART_PER_KIND, seed=SEED)
for item in chart_items:
    item["file"] = f"charts/{item['file']}"
print(len(chart_items), "biểu đồ;", Counter((i["chart_type"], i["values_labelled"]) for i in chart_items))

## 6. Ghi `dataset.json` và xem thử

In [ ]:
import datetime
dataset = {
    "meta": {
        "created": datetime.datetime.now().isoformat(timespec="seconds"), "seed": SEED,
        "data_dir": DATA_DIR,
        "counts": {"coco": len(coco_items), "scicap": len(scicap_items), "charts": len(chart_items)},
        "sources": {
            "coco_val2017": "COCO Captions [18], ảnh val2017, giấy phép theo từng ảnh",
            "scicap_plus_test": "SciCap+ [40], tập test, CC BY-NC 4.0",
            "synthetic_chart": "Biểu đồ tự vẽ bằng matplotlib, có đáp án",
        },
    },
    "items": coco_items + scicap_items + chart_items,
}
path = os.path.join(DATA_DIR, "dataset.json")
with open(path, "w", encoding="utf8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=1)
print("Đã ghi", path, dataset["meta"]["counts"])

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

preview = [coco_items[0], coco_items[1], scicap_items[0], scicap_items[1], chart_items[0], chart_items[-1]]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, item in zip(axes.ravel(), preview):
    ax.imshow(Image.open(os.path.join(DATA_DIR, item["file"])))
    ax.set_title(item["id"][:40], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Tải `dataset.json` về máy để gửi lại

In [ ]:
from google.colab import files
files.download(os.path.join(DATA_DIR, "dataset.json"))